# Credit Risk Feature Engineering

## Objective

This notebook transforms the raw Home Credit application data into a modeling-ready dataset.

The objectives are to:

1. Split the data before learning preprocessing parameters
2. Correct invalid and special values
3. Create interpretable credit-risk features
4. Define numerical and categorical feature groups
5. Prepare a reproducible preprocessing strategy
6. Avoid target leakage

This notebook focuses on feature engineering and preprocessing design. Model training and evaluation will be completed in the next notebook.

## Modeling Principles

The feature-engineering workflow follows several important principles:

- Split the data before fitting imputers, encoders, or scalers
- Use only information available at application time
- Preserve missingness when it may carry risk information
- Prefer interpretable financial ratios over arbitrary transformations
- Handle division-by-zero explicitly
- Keep raw variables when engineered features provide complementary information
- Apply identical transformations to training and test data

## 1. Setup and Data Loading

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: /Users/hxxy/Desktop/找工/credit-risk-decisioning/.venv/bin/python
Python version: 3.13.1 (main, Dec  3 2024, 17:59:52) [Clang 16.0.0 (clang-1600.0.26.4)]


In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = RAW_DATA_DIR / "application_train.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRAIN_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

### Basic Validation

Before feature engineering, verify the target, identifier, and dataset grain.

In [3]:
required_columns = ["SK_ID_CURR", "TARGET"]

missing_required_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Missing required columns: {missing_required_columns}"
    )

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Unique applicants:", df["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", df["SK_ID_CURR"].duplicated().sum())
print("\nTarget distribution:")
print(df["TARGET"].value_counts(dropna=False))
print("\nTarget rate:")
print(df["TARGET"].mean())

Rows: 307511
Columns: 122
Unique applicants: 307511
Duplicate applicant IDs: 0

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Target rate:
0.08072881945686496


### Observation

- The application table contains one row per applicant.
- `SK_ID_CURR` is an identifier and should not be used as a model feature.
- `TARGET = 1` represents applicants with repayment difficulties.
- The target is imbalanced, so model evaluation should not rely on accuracy alone.

## 2. Train/Test Split

The dataset is split before fitting imputers, encoders, scalers, or other preprocessing objects.

A stratified split is used to preserve the target distribution in both datasets.

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype("int8")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining target rate:")
print(y_train.mean())

print("\nTest target rate:")
print(y_test.mean())

X_train shape: (246008, 121)
X_test shape: (61503, 121)

Training target rate:
0.08072908198107379

Test target rate:
0.08072776937710356


In [5]:
assert len(X_train) + len(X_test) == len(df)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert abs(y_train.mean() - y_test.mean()) < 0.001

print("Train/test split validation passed.")

Train/test split validation passed.


In [6]:
train_ids = X_train["SK_ID_CURR"].copy()
test_ids = X_test["SK_ID_CURR"].copy()

X_train = X_train.drop(columns=["SK_ID_CURR"])
X_test = X_test.drop(columns=["SK_ID_CURR"])

print("Modeling training shape:", X_train.shape)
print("Modeling test shape:", X_test.shape)

Modeling training shape: (246008, 120)
Modeling test shape: (61503, 120)


## 3. Initial Feature Scope

The first feature-engineering version focuses on variables that are:

- available at application time,
- interpretable to business stakeholders,
- supported by the EDA and data dictionary,
- suitable for an interpretable baseline model.

Additional variables can be added after the baseline workflow is validated.

In [7]:
initial_features = [
    # Application and demographics
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    
    # Income and credit
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    
    # Employment and applicant profile
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    
    # Time-based variables
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    
    # Geographic / application context
    "REGION_POPULATION_RELATIVE",
    "REGION_RATING_CLIENT",
    "REGION_RATING_CLIENT_W_CITY",
    
    # External credit scores
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    
    # Bureau inquiries
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

In [8]:
missing_initial_features = [
    col for col in initial_features
    if col not in X_train.columns
]

if missing_initial_features:
    raise ValueError(
        f"Initial features missing from dataset: {missing_initial_features}"
    )

print("Number of initial features:", len(initial_features))

Number of initial features: 32


In [9]:
X_train_fe = X_train[initial_features].copy()
X_test_fe = X_test[initial_features].copy()

print("Training feature dataset:", X_train_fe.shape)
print("Test feature dataset:", X_test_fe.shape)

Training feature dataset: (246008, 32)
Test feature dataset: (61503, 32)


In [10]:
feature_summary = pd.DataFrame({
    "dtype": X_train_fe.dtypes.astype(str),
    "missing_count": X_train_fe.isna().sum(),
    "missing_rate": X_train_fe.isna().mean(),
    "n_unique": X_train_fe.nunique(dropna=False)
}).sort_values(
    by="missing_rate",
    ascending=False
)

feature_summary

,dtype,missing_count,missing_rate,n_unique
EXT_SOURCE_1,float64,138595,0.563376,94565
OCCUPATION_TYPE,str,76940,0.312754,19
EXT_SOURCE_3,float64,48805,0.198388,807
AMT_REQ_CREDIT_BUREAU_YEAR,float64,33244,0.135134,25
AMT_REQ_CREDIT_BUREAU_QRT,float64,33244,0.135134,11
AMT_REQ_CREDIT_BUREAU_MON,float64,33244,0.135134,24
AMT_REQ_CREDIT_BUREAU_WEEK,float64,33244,0.135134,10
AMT_REQ_CREDIT_BUREAU_DAY,float64,33244,0.135134,10
AMT_REQ_CREDIT_BUREAU_HOUR,float64,33244,0.135134,6
EXT_SOURCE_2,float64,531,0.002158,108826


In [11]:
categorical_features_raw = (
    X_train_fe
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

numerical_features_raw = (
    X_train_fe
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

print("Categorical features:", len(categorical_features_raw))
print(categorical_features_raw)

print("\nNumerical features:", len(numerical_features_raw))
print(numerical_features_raw)

Categorical features: 10
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

Numerical features: 22
['CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']


/var/folders/qk/7q9gfxws52n7zsd9_0fr1_wh0000gn/T/ipykernel_90321/151657628.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category"])


### Initial Feature Review

The selected features include a mixture of numerical and categorical variables.

Several important risk variables contain missing values, especially the external credit scores and occupation information. Missing values will not be dropped automatically because missingness may itself carry information about applicant risk.

Time variables are currently stored as negative day counts and must be converted into interpretable units before modeling.